# Bangla ASR test: `hishab/titu_stt_bn_conformer_large`

NeMo **Conformer-CTC Large**, fine-tuned on ~4.4K hours of Bangla. Model card: **~7.9% avg WER / 1.75% CER** (Common Voice 6.5%, Fleurs 10.5%, Kathbath 9.8%). Size ~487 MB.

**License: CC-BY-NC-4.0 (non-commercial only).**

What this notebook does:
1. installs NeMo
2. downloads the model from Hugging Face
3. lets you upload your own audio (mp3/wav/flac/ogg/m4a) or read it from Google Drive
4. transcribes on **CPU** (same as your server) or GPU, and prints transcript, timing, RTF
5. computes **WER / CER** if you also upload a reference `.txt` with the same file name
6. shows every error with its full traceback and saves results you can download

> **RTF** = processing time / audio duration. Below 1.0 is faster than real time.
> Use **Runtime > Change runtime type > CPU** to measure CPU speed realistically.

## 1. Settings

In [ ]:
DEVICE = "cpu"        # "cpu" to mimic your CPU server, or "cuda" if you picked a GPU runtime
THREADS = 0           # CPU threads for torch, 0 = all available cores
CHUNK_SEC = 30        # split long audio into chunks of N seconds (0 = no chunking)
BATCH_SIZE = 4        # chunks per forward pass

AUDIO_SOURCE = "upload"   # "upload" = pick files from your PC, "drive" = read a Google Drive folder
DRIVE_AUDIO_DIR = "/content/drive/MyDrive/sample-audio"   # used when AUDIO_SOURCE == "drive"
INCLUDE_MODEL_TEST_WAV = True   # also transcribe the sample clip shipped with the model

## 2. Install dependencies
Takes ~3-5 minutes. If the import cell below fails with a numpy / binary error,
use **Runtime > Restart session** and run from section 3 onward (no need to reinstall).

In [ ]:
!apt-get -qq install -y ffmpeg libsndfile1 > /dev/null
!pip install -q "nemo_toolkit[asr]" huggingface_hub librosa soundfile jiwer

## 3. Imports and environment check

In [ ]:
import os, sys, time, json, glob, shutil, tempfile, traceback, logging
from pathlib import Path

import numpy as np
import torch
import librosa
import soundfile as sf

import nemo
import nemo.collections.asr as nemo_asr
logging.getLogger("nemo_logger").setLevel(logging.ERROR)   # NeMo is very chatty

if DEVICE == "cuda" and not torch.cuda.is_available():
    print("WARNING: CUDA requested but no GPU in this runtime, falling back to CPU")
    DEVICE = "cpu"
if THREADS:
    torch.set_num_threads(THREADS)

print("python :", sys.version.split()[0])
print("torch  :", torch.__version__)
print("nemo   :", nemo.__version__)
print("device :", DEVICE, "| torch threads:", torch.get_num_threads(), "| cpu cores:", os.cpu_count())
!lscpu | grep -E "Model name|^CPU\(s\)"

## 4. Download the model

In [ ]:
os.environ["HF_HUB_DISABLE_XET"] = "1"   # Xet transfer can stall; plain HTTP is reliable
from huggingface_hub import hf_hub_download

REPO_ID = "hishab/titu_stt_bn_conformer_large"
MODEL_DIR = Path("/content/model")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

t0 = time.time()
MODEL_PATH = hf_hub_download(REPO_ID, "titu_stt_bn_conformer_large.nemo", local_dir=MODEL_DIR)
TEST_WAV = hf_hub_download(REPO_ID, "test_bn_conformer.wav", local_dir=MODEL_DIR)
print(f"Downloaded in {time.time()-t0:.1f}s")
!ls -lh /content/model

## 5. Load the model

In [ ]:
t0 = time.time()
model = nemo_asr.models.ASRModel.restore_from(MODEL_PATH, map_location=DEVICE)
model = model.to(DEVICE).eval()
LOAD_SEC = time.time() - t0
print(f"Loaded {type(model).__name__} in {LOAD_SEC:.1f}s | "
      f"params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

## 6. Get your audio
* **upload**: a file picker opens. Select your audio files, plus optional reference
  transcripts named like the audio (`call.mp3` + `call.txt`) to get WER/CER.
* **drive**: mounts Google Drive and reads `DRIVE_AUDIO_DIR`.

In [ ]:
AUDIO_DIR = Path("/content/sample-audio")
AUDIO_DIR.mkdir(exist_ok=True)
AUDIO_EXTS = {".wav", ".mp3", ".flac", ".ogg", ".m4a", ".opus", ".webm"}

if AUDIO_SOURCE == "upload":
    from google.colab import files
    for name, data in files.upload().items():
        (AUDIO_DIR / name).write_bytes(data)
elif AUDIO_SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    for p in Path(DRIVE_AUDIO_DIR).iterdir():
        if p.suffix.lower() in AUDIO_EXTS | {".txt"}:
            shutil.copy(p, AUDIO_DIR / p.name)

if INCLUDE_MODEL_TEST_WAV:
    shutil.copy(TEST_WAV, AUDIO_DIR / Path(TEST_WAV).name)

AUDIO_FILES = sorted(p for p in AUDIO_DIR.iterdir() if p.suffix.lower() in AUDIO_EXTS)
print(f"{len(AUDIO_FILES)} audio file(s):")
for p in AUDIO_FILES:
    ref = " (+ reference .txt)" if p.with_suffix(".txt").exists() else ""
    print(" -", p.name + ref)

## 7. Helper functions

In [ ]:
SR = 16000

def load_audio(path):
    audio, _ = librosa.load(str(path), sr=SR, mono=True)   # any format -> 16 kHz mono
    if audio.size == 0:
        raise ValueError("audio decoded to 0 samples")
    return audio, len(audio) / SR

def split_chunks(audio, chunk_sec):
    size = int(chunk_sec * SR)
    if size <= 0 or len(audio) <= size:
        return [audio]
    chunks = [audio[i:i + size] for i in range(0, len(audio), size)]
    if len(chunks) > 1 and len(chunks[-1]) < SR:          # merge tiny (<1 s) tail
        chunks[-2] = np.concatenate([chunks[-2], chunks.pop()])
    return chunks

def hyp_to_text(h):
    if isinstance(h, str):
        return h
    if hasattr(h, "text"):
        return h.text or ""
    if isinstance(h, (list, tuple)) and h:
        return hyp_to_text(h[0])
    return str(h)

def transcribe(audio, chunk_sec=CHUNK_SEC):
    chunks = split_chunks(audio, chunk_sec)
    with tempfile.TemporaryDirectory() as tmp:
        paths = []
        for i, c in enumerate(chunks):
            p = f"{tmp}/chunk_{i:04d}.wav"
            sf.write(p, c, SR, subtype="PCM_16")
            paths.append(p)
        with torch.inference_mode():
            out = model.transcribe(paths, batch_size=BATCH_SIZE, verbose=False)
    if isinstance(out, tuple):          # hybrid models return (best, all)
        out = out[0]
    return " ".join(t for t in (hyp_to_text(h).strip() for h in out) if t), len(chunks)

## 8. Run the test

In [ ]:
import jiwer

results, errors = [], []
for f in AUDIO_FILES:
    print("=" * 80)
    print("FILE:", f.name)
    rec = {"file": f.name, "status": "ok"}
    try:
        audio, dur = load_audio(f)
        t0 = time.time()
        text, n_chunks = transcribe(audio)
        infer = time.time() - t0
        rec.update(duration_sec=round(dur, 2), chunks=n_chunks, inference_sec=round(infer, 2),
                   rtf=round(infer / dur, 3), text=text)
        print(f"duration {dur:.2f}s | chunks {n_chunks} | inference {infer:.2f}s | RTF {infer/dur:.3f}")
        print("TRANSCRIPT:\n" + (text or "<empty>"))

        ref_file = f.with_suffix(".txt")
        if ref_file.exists():
            ref = ref_file.read_text(encoding="utf-8").strip()
            rec.update(reference=ref, wer=round(jiwer.wer(ref, text), 4), cer=round(jiwer.cer(ref, text), 4))
            print(f"WER {rec['wer']:.2%} | CER {rec['cer']:.2%}")
    except Exception as e:
        rec.update(status="error", error=f"{type(e).__name__}: {e}")
        errors.append((f.name, traceback.format_exc()))
        print("ERROR:", rec["error"])
    results.append(rec)

## 9. Summary

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 120)
df = pd.DataFrame(results)
display(df.drop(columns=[c for c in ["reference"] if c in df]))

ok = df[df.status == "ok"] if "status" in df else df
if len(ok):
    print(f"\nModel load {LOAD_SEC:.1f}s | total audio {ok.duration_sec.sum():.1f}s | "
          f"total inference {ok.inference_sec.sum():.1f}s | overall RTF {ok.inference_sec.sum()/ok.duration_sec.sum():.3f}")
print(f"{len(ok)} ok, {len(errors)} failed")

## 10. Errors (full tracebacks)

In [ ]:
if not errors:
    print("No errors")
for name, tb in errors:
    print("=" * 80)
    print("FILE:", name)
    print(tb)

## 11. Save and download results

In [ ]:
from datetime import datetime
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
out_json = f"/content/titu_stt_bn_conformer_large_results_{stamp}.json"
with open(out_json, "w", encoding="utf-8") as fh:
    json.dump({"model": REPO_ID, "device": DEVICE, "threads": torch.get_num_threads(),
               "chunk_sec": CHUNK_SEC, "load_sec": round(LOAD_SEC, 2),
               "results": results, "errors": [{"file": n, "traceback": t} for n, t in errors]},
              fh, ensure_ascii=False, indent=2)
print("Saved", out_json)

from google.colab import files
files.download(out_json)